# Team Game Logs — Rolling Feature Correlation EDA (2024–25)

**Objetivo:** analizar correlaciones de Pearson entre los resultados del partido actual y los promedios móviles de los últimos 10 partidos, evitando *data leakage* y resaltando tendencias recientes de rendimiento.

Este notebook replica la exploración original pero limita las features a columnas `ROLL10_`, garantizando que solo se use información disponible antes de cada encuentro.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.impute import SimpleImputer


In [ ]:
parquet_path = "/Users/pablo/Documents/BigData/BasketballAnalysis/00_data/00c_final/2024-25/teamgamelogs_by_game.parquet"

try:
    df = pd.read_parquet(parquet_path, engine="pyarrow")
except ImportError as exc:
    raise ImportError("Se requiere pyarrow para leer el archivo parquet. Instale el paquete pyarrow e intente de nuevo.") from exc

print(f"Dataset shape: {df.shape}")
df.head(3)


## Información de columnas y valores nulos

Se muestran los tipos de datos y el porcentaje de valores nulos por columna (máximo 30 columnas con más nulos). No se realizan imputaciones en esta etapa. Para cada cálculo de correlación se aplicará `dropna()` únicamente sobre las columnas implicadas, minimizando sesgos por registros incompletos.

In [ ]:
if "GAME_DATE" in df.columns:
    df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"], errors="coerce")

df.info()

null_summary = (
    pd.DataFrame({
        "null_count": df.isna().sum(),
        "null_pct": df.isna().mean() * 100,
    })
    .sort_values(by=["null_pct", "null_count"], ascending=[False, False])
    .head(30)
)
null_summary


In [ ]:
# 1. Verificar la columna WL
wl_col = df.get("WL")
print(f"Valores únicos en WL: {df['WL'].unique() if wl_col is not None else 'COLUMNA NO EXISTE'}")

if wl_col is not None:
    df["WL_NUM"] = (
        wl_col.astype(str)
        .str.strip()
        .str.upper()
        .map({"W": 1, "L": 0, "NAN": np.nan})  # Manejar nulos explícitamente
    )
    print(f"Distribución WL_NUM:\n{df['WL_NUM'].value_counts(dropna=False)}")
else:
    df["WL_NUM"] = np.nan

# 2. Selección exclusiva de rolling features
exclude_columns = {
    "SEASON_YEAR", "TEAM_ID", "TEAM_ABBREVIATION", "TEAM_NAME",
    "GAME_ID", "GAME_DATE", "MATCHUP", "WL", "WL_NUM",
    "endpoint", "season", "game_id", "AVAILABLE_FLAG"
}

rolling_features = [col for col in df.columns if col.startswith('ROLL10_')]
exclude_rolling = ['ROLL10_PLUS_MINUS', 'ROLL10_PTS']
valid_features = [f for f in rolling_features if f not in exclude_rolling]

print(f"Total rolling features detectadas: {len(rolling_features)}")
print(f"Rolling features válidas tras exclusiones: {len(valid_features)}")
print(f"Ejemplo de features: {valid_features[:10]}")

# 3. Validaciones anti-leakage
print("=== ANÁLISIS CON ROLLING AVERAGES (Sin Data Leakage) ===")
print("FEATURES: Rolling averages últimos 10 partidos (t-10 hasta t-1)")
print("TARGETS:  Resultados del partido actual (t)")
print("VENTAJA:  Solo información disponible ANTES del partido")

current_game_features = [col for col in valid_features if not col.startswith('ROLL10_')]
assert len(current_game_features) == 0, f"DATA LEAKAGE DETECTADO: {current_game_features}"
assert 'WL_NUM' not in valid_features
assert 'PLUS_MINUS' not in valid_features
assert 'PTS' not in valid_features

rank_exclusions = {col for col in df.columns if col.endswith('_RANK')}
all_exclusions = exclude_columns.union(rank_exclusions)
print(f"Columnas administrativas excluidas: {len(all_exclusions)}")


In [ ]:
targets = ["WL_NUM", "PLUS_MINUS", "PTS"]

for target in targets:
    if target not in df.columns:
        print(f"Objetivo '{target}' no encontrado. Se omite el análisis de correlación.")
        continue

    available_features = [feat for feat in valid_features if feat != target]
    if not available_features:
        print(f"No hay features válidas disponibles para correlacionar con {target}.")
        continue

    print(f"\n=== ANÁLISIS PARA {target} ===")
    print(f"Features disponibles: {len(available_features)}")

    subset_df = df[available_features + [target]].copy()

    print(f"Filas totales: {len(subset_df)}")
    print(f"Filas con target no nulo: {subset_df[target].notna().sum()}")

    X = subset_df[available_features]
    y = subset_df[target]
    valid_target_mask = y.notna()

    if valid_target_mask.sum() == 0:
        print(f"No hay valores válidos para el target {target}.")
        continue

    X_valid = X[valid_target_mask]
    y_valid = y[valid_target_mask]

    imputer = SimpleImputer(strategy='median')
    X_imputed = imputer.fit_transform(X_valid)
    X_imputed = pd.DataFrame(X_imputed, columns=available_features, index=X_valid.index)

    processed_df = pd.concat([X_imputed, y_valid], axis=1)

    print(f"Filas después de procesamiento: {len(processed_df)}")

    corr_matrix = processed_df.corr(method="pearson")

    if target not in corr_matrix.columns:
        print(f"No se pudo calcular correlación para {target}")
        continue

    corr_series = corr_matrix[target].drop(labels=[target], errors='ignore')
    corr_series = corr_series.dropna().sort_values(ascending=False)

    print(f"Correlaciones calculadas: {len(corr_series)}")

    if corr_series.empty:
        print(f"No se encontraron correlaciones para {target}")
        continue

    significant_corr = corr_series[corr_series.abs() > 0.1]

    if significant_corr.empty:
        print("No hay correlaciones significativas (|r| > 0.1). Se mostrarán las top 20 absolutas.")
        significant_corr = corr_series.reindex(corr_series.abs().sort_values(ascending=False).index[:20])

    positive_corr = significant_corr[significant_corr > 0].head(15)
    negative_corr = significant_corr[significant_corr < 0].sort_values().head(10)

    print(f"Correlación máxima: {corr_series.max():.3f}")
    print(f"Correlación mínima: {corr_series.min():.3f}")
    print(f"Correlación promedio (abs): {corr_series.abs().mean():.3f}")

    if not corr_series.empty:
        top_abs_feature = corr_series.abs().idxmax()
        print(f"Mayor correlación absoluta: {top_abs_feature} ({corr_series[top_abs_feature]:.3f})")

    if not positive_corr.empty:
        print(f"\nTop {len(positive_corr)} correlaciones positivas:")
        display(positive_corr.to_frame(name="pearson_correlation"))

        plt.figure(figsize=(max(12, len(positive_corr) * 0.8), 6))
        positive_corr.plot(kind="bar")
        plt.title(f"Top Rolling Averages correlacionados con {target}\n(Ventana: 10 partidos anteriores)")
        plt.ylabel("Correlación de Pearson")
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("No se encontraron correlaciones positivas significativas.")

    if not negative_corr.empty:
        print(f"\nTop {len(negative_corr)} correlaciones negativas:")
        display(negative_corr.to_frame(name="pearson_correlation"))

        plt.figure(figsize=(max(12, len(negative_corr) * 0.8), 6))
        negative_corr.plot(kind="bar")
        plt.title(f"Bottom Rolling Averages correlacionados con {target}\n(Ventana: 10 partidos anteriores)")
        plt.ylabel("Correlación de Pearson")
        plt.axhline(y=0, color='r', linestyle='-', alpha=0.3)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        print("No se encontraron correlaciones negativas significativas.")

    null_info = subset_df[available_features].isnull().sum()
    high_null_cols = null_info[null_info > 0]
    if not high_null_cols.empty:
        print(f"Features con nulos: {len(high_null_cols)}")
        print("Top 10 features con más nulos:")
        print(high_null_cols.sort_values(ascending=False).head(10))


In [ ]:
if valid_features:
    variances = df[valid_features].var(numeric_only=True).dropna().sort_values(ascending=False)
    top_var_features = variances.head(30).index.tolist()

    if top_var_features:
        feature_corr_df = df[top_var_features].dropna()
        if not feature_corr_df.empty and len(top_var_features) > 1:
            feature_corr_matrix = feature_corr_df.corr(method="pearson")

            plt.figure(figsize=(18, 14))
            im = plt.imshow(feature_corr_matrix, aspect="auto", vmin=-1, vmax=1)
            plt.colorbar(im, fraction=0.046, pad=0.04)
            plt.xticks(ticks=range(len(top_var_features)), labels=top_var_features, rotation=90)
            plt.yticks(ticks=range(len(top_var_features)), labels=top_var_features)
            plt.title("Matriz de correlación (Rolling averages - Top 30 varianzas)")
            plt.tight_layout()
            plt.show()
        else:
            print("No hay suficientes datos completos para calcular la matriz de correlación de rolling features.")
    else:
        print("No hay rolling features con varianza suficiente para el análisis de correlación.")
else:
    print("No hay rolling features válidas para analizar.")


## Nota

Si alguna correlación aparece como `NaN`, se debe a columnas constantes o sin observaciones válidas después del filtrado. Esto se mitiga excluyendo columnas sin variabilidad y aplicando imputación de mediana por bloque de análisis.